# 3주차: 팀 위키로 답하는 RAG 노트북

`로드 → 분할 → 임베딩 → 저장 → 검색 → 생성 → 평가` 순서로 한 섹션씩 실행해요. 각 섹션 끝의 **📝 관찰 메모**는 직접 채워요. 이 메모가 위키 `내 정리`와 발표 재료가 돼요.

**사전 준비**
1. 저장소 루트에서 `python scripts/merge_wiki.py` (팀 위키 최신화)
2. 이 폴더(`members/mingi/week3/`)에서 가상환경 + `pip install -r requirements.txt`
3. `.env`에 `GEMINI_API_KEY=...` (2주차와 같은 키), 커널은 `.venv` 선택

In [27]:
import sys
print(sys.executable)   # .venv 안의 python 경로가 나와야 해요

from pathlib import Path
import os
from dotenv import load_dotenv
ENV_PATH = Path.cwd() / ".env"                 # 노트북과 같은 폴더(week3)의 .env
print(".env 위치:", ENV_PATH, "있음" if ENV_PATH.exists() else "없음")
load_dotenv(ENV_PATH, override=True)           # .env 값을 os.environ에 넣어요
print("GEMINI_API_KEY 불러옴:", bool(os.getenv("GEMINI_API_KEY")))

REPO = Path("../../..").resolve()   # 저장소 루트
WIKI = REPO / "team" / "wiki"
print(REPO, WIKI.exists())

c:\Users\ckawo\AppData\Local\anaconda3\envs\study\python.exe
.env 위치: c:\Users\ckawo\OneDrive\바탕 화면\projects\Study-service_basic\members\mingi\week3\.env 있음
GEMINI_API_KEY 불러옴: True
C:\Users\ckawo\OneDrive\바탕 화면\projects\Study-service_basic True


## 1. 로드

팀 위키(`team/wiki/*.md`)를 Document로 불러와요. `index.md`, `_template.md`는 빼요.
- 확인: 페이지 수(지금은 15개), `metadata["source"]`

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(str(WIKI), glob="*.md", loader_cls=TextLoader,
                         loader_kwargs={"encoding": "utf-8"})
docs = [d for d in loader.load()
        if Path(d.metadata["source"]).name not in ("index.md", "_template.md")]

# merge_wiki.py가 맨 위에 넣은 자동 생성 표시 줄은 검색 대상에서 빼요 (evaluate.py와 같은 처리)
for d in docs:
    d.page_content = "\n".join(l for l in d.page_content.splitlines()
                               if not l.startswith("<!-- merge_wiki.py"))

print("페이지 수:", len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:300])

C:\Users\ckawo\AppData\Local\Temp\ipykernel_41364\177305085.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


페이지 수: 15
{'source': 'C:\\Users\\ckawo\\OneDrive\\바탕 화면\\projects\\Study-service_basic\\team\\wiki\\1-docker-concepts_sejin.md'}

# Docker 핵심 개념: 이미지, 컨테이너, 레이어, 레지스트리

## 한 줄 정의

Dockerfile(설계서)대로 설치까지 끝내 묶은 결과물이 이미지, 그 이미지를 실행한 격리된 프로세스가 컨테이너, 이미지를 이루는 Dockerfile 한 줄 한 줄의 층이 레이어, 이미지를 올리고 받는 원격 창고가 레지스트리다.

## 왜 필요한가

파이썬 버전, 패키지, OS가 조금만 달라도 "내 컴퓨터에선 되는데 다른 컴퓨터에선 안 되는" 문제가 생긴다. Docker는 프로그램과 실행 환경을 이미지 하나로 묶어서, Docker만 있으면


## 2. 분할 (+ 청크 ID)

기준선 설정 `chunk_size=500`, `chunk_overlap=50`으로 자르고, 청크마다 `페이지이름-번호` ID를 붙여요.
청크 ID는 질문, 정답, 검색 결과를 잇는 열쇠예요.
- 확인: 청크 수, 가장 짧은/긴 청크, 청크 하나의 metadata

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE, CHUNK_OVERLAP = 500, 50
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

chunks = []
for doc in docs:
    page = Path(doc.metadata["source"]).stem          # 예: docker-compose_mingi
    for i, c in enumerate(splitter.split_documents([doc])):
        c.metadata["page"] = page
        c.metadata["id"] = f"{page}-{i}"
        chunks.append(c)

print("청크 수:", len(chunks))
print(chunks[0].metadata)

청크 수: 109
{'source': 'C:\\Users\\ckawo\\OneDrive\\바탕 화면\\projects\\Study-service_basic\\team\\wiki\\1-docker-concepts_sejin.md', 'page': '1-docker-concepts_sejin', 'id': '1-docker-concepts_sejin-0'}


In [22]:
chunks[3]

Document(metadata={'source': 'C:\\Users\\ckawo\\OneDrive\\바탕 화면\\projects\\Study-service_basic\\team\\wiki\\1-docker-concepts_sejin.md', 'page': '1-docker-concepts_sejin', 'id': '1-docker-concepts_sejin-3'}, page_content='- 만들면 내용으로 계산한 ID가 붙고, 보통은 이름:태그(예: week1-app:latest)로 불러 쓴다.\n- 한 번 만들면 바꿀 수 없다. 고치려면 Dockerfile이나 코드를 고쳐 다시 빌드한다. 안 바뀐 레이어는 재사용되고, 새 ID의 이미지가 생긴다.\n- 공유: 레지스트리(Docker Hub 등)에 push하면 다른 사람이 pull해서 똑같이 쓸 수 있다.\n  - 이미지를 넘기면 받은 사람은 빌드 없이 완전히 같은 결과를 실행한다(크다: 수백 MB).\n  - Dockerfile을 넘기면 작고 읽기 쉽지만, 받은 사람이 직접 빌드해야 하고 빌드 시점에 따라 결과가 달라질 수 있다(베이스 이미지 갱신, 버전 안 박은 패키지).\n  - 실무에서는 Dockerfile과 코드는 Git에(원본), 이미지는 레지스트리에(배포용) 둔다.\n  - 이미지 안의 파일은 받은 사람이 다 꺼내 볼 수 있으니, 올리기 전에 .env 같은 비밀이 빠졌는지 확인한다.\n\n### 2. 컨테이너')

In [5]:
lengths = sorted((len(c.page_content), c.metadata["id"]) for c in chunks)
print("가장 짧은 청크:", lengths[:3])
print("가장 긴 청크:", lengths[-3:])
print("평균 길이:", sum(l for l, _ in lengths) / len(lengths))

shortest = min(chunks, key=lambda c: len(c.page_content))
print("\n--- 가장 짧은 청크 내용 ---")
print(shortest.page_content)

가장 짧은 청크: [(8, 'api-key-env_mingi-4'), (8, 'async-fastapi_mingi-4'), (8, 'container-commands_mingi-4')]
가장 긴 청크: [(495, '1-docker-concepts_sejin-3'), (496, '2-fastapi-redis-async_sejin-0'), (496, 'volume-port_mingi-0')]
평균 길이: 348.41284403669727

--- 가장 짧은 청크 내용 ---
## 출처

-


In [23]:
page, n = shortest.metadata["id"].rsplit("-", 1)
prev = next(c for c in chunks if c.metadata["id"] == f"{page}-{int(n) - 1}")
last_para = prev.page_content.split("\n\n")[-1]
print("직전 청크 길이:", len(prev.page_content))
print("직전 청크 마지막 문단 길이:", len(last_para))
print(repr(last_para[-100:]))

직전 청크 길이: 490
직전 청크 마지막 문단 길이: 490
' rm --cached .env`가 필요해요.\n- 이번 주에 파일 이름이 `env`(점 없음)였을 때 무시도, 로드도 안 됐던 경험을 "내가 막힌 점"에 적으면 좋은 사례가 돼요.'


**📝 관찰 메모 (2. 분할)**
- 가장 짧은 청크는 왜 그렇게 잘렸나?
- 내 위키의 `### Q.` 제목 단위로 자른다면 결과가 어떻게 달라질까?
-

## 3. 임베딩 (sentence-transformers)

`BAAI/bge-m3`를 sentence-transformers로 한 번만 불러와서, 데모와 벡터 DB 저장에 같이 써요. (첫 실행 때 약 2GB를 내려받아요)

발표 데모 순서
1. `print(model)`: 모델이 몇 단계(모듈)로 이어져 있나? Pooling은 어떤 방식인가?
2. 토큰 벡터(여러 개) → 문장 벡터(한 개)
3. 문장 5개의 유사도 표: 예상과 비교

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")
print(model)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


In [7]:
# L2에서 고른 문장 5개로 바꿔도 돼요: 비슷한 뜻 2개, 단어는 겹치지만 뜻이 다른 것 1개, 완전히 다른 것 2개
sentences = [
    "컨테이너가 꺼져도 파일을 남기려면 볼륨을 연결한다.",
    "도커 실행이 끝난 뒤에도 결과물을 보존하는 방법은 -v 옵션이다.",
    "컨테이너 배송 일정이 태풍 때문에 늦어졌다.",
    "같은 질문은 Redis에 저장한 답을 바로 돌려준다.",
    "오늘 점심은 김치찌개를 먹었다.",
]

tok = model.encode(sentences[0], output_value="token_embeddings")
print("토큰 벡터:", tuple(tok.shape), " <- 토큰 수 x 차원")

emb = model.encode(sentences, normalize_embeddings=True)
print("문장 벡터:", emb.shape, " <- 문장 수 x 차원")

토큰 벡터: (23, 1024)  <- 토큰 수 x 차원
문장 벡터: (5, 1024)  <- 문장 수 x 차원


In [8]:
import pandas as pd

sim = model.similarity(emb, emb)          # 정규화했으므로 코사인 유사도
labels = [f"s{i}" for i in range(len(sentences))]
pd.DataFrame(sim.numpy(), index=labels, columns=labels).round(3)

,s0,s1,s2,s3,s4
s0,1.000,0.562,0.465,0.492,0.357
s1,0.562,1.000,0.307,0.447,0.270
s2,0.465,0.307,1.000,0.306,0.425
s3,0.492,0.447,0.306,1.000,0.335
s4,0.357,0.270,0.425,0.335,1.000


In [9]:
# L4 실험: 정규화한 벡터에서 코사인, 내적, L2 거리의 관계
import numpy as np

a, b = emb[0], emb[1]
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
dot = a @ b
l2 = np.linalg.norm(a - b)
print(f"cos={cos:.4f}  dot={dot:.4f}  L2={l2:.4f}  2-2cos={2 - 2 * cos:.4f}  L2^2={l2 ** 2:.4f}")

cos=0.5621  dot=0.5621  L2=0.9358  2-2cos=0.8758  L2^2=0.8758


**📝 관찰 메모 (3. 임베딩)**
- `print(model)`의 각 단계는 무슨 일을 하나? bge-m3의 pooling 방식은?
- 유사도 표에서 예상과 다르게 나온 쌍과 그 이유는?
- 정규화하면 cos, dot, L2 순위가 왜 같아지나?
-

## 4. 저장 (Chroma)

위에서 불러온 `model`을 LangChain이 쓸 수 있게 감싸서(`Embeddings` 인터페이스) Chroma에 넣어요.
거리는 `cosine`으로 정해요. 아래에서 기본값(L2)과 점수를 비교해 봐요.

In [10]:
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma

class STEmbeddings(Embeddings):
    """이미 불러온 SentenceTransformer를 LangChain에서 쓰기 위한 얇은 래퍼"""
    def __init__(self, model):
        self.model = model
    def embed_documents(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()
    def embed_query(self, text):
        return self.model.encode(text, normalize_embeddings=True).tolist()

embeddings = STEmbeddings(model)

vectorstore = Chroma.from_documents(
    chunks, embeddings,
    ids=[c.metadata["id"] for c in chunks],
    collection_name="wiki-cosine",
    collection_metadata={"hnsw:space": "cosine"},
)
print("저장된 청크 수:", vectorstore._collection.count())

저장된 청크 수: 109


In [24]:
# L4 실험: 거리 설정을 안 했을 때(기본값)와 cosine일 때 점수 비교
store_default = Chroma.from_documents(
    chunks, embeddings,
    ids=[c.metadata["id"] for c in chunks],
    collection_name="wiki-default",
)

q = "컨테이너가 꺼진 뒤에도 결과 파일을 남기려면?"
for name, store in [("기본값", store_default), ("cosine", vectorstore)]:
    print(f"--- {name} ---")
    for doc, score in store.similarity_search_with_score(q, k=3):
        print(f"{score:.4f}  {doc.metadata['id']}")

--- 기본값 ---
0.7128  container-commands_mingi-2
0.7576  1-docker-concepts_sejin-2
0.8226  1-docker-concepts_sejin-4
--- cosine ---
0.3564  container-commands_mingi-2
0.3788  1-docker-concepts_sejin-2
0.4113  1-docker-concepts_sejin-4


**📝 관찰 메모 (4. 저장)**
- Chroma의 기본 거리는 무엇이었나? 점수는 클수록 가까운가, 작을수록 가까운가?
- 두 설정의 순위는 같았나? 점수 값은 어떤 관계였나?
-

## 5. 검색 (retriever)

질문을 넣으면 가까운 청크 k개를 돌려주는 retriever를 만들어요. 기준선은 `k=5`예요.
- 확인: 질문 2~3개로 top-5 청크 ID와 점수

In [25]:
K = 5
retriever = vectorstore.as_retriever(search_kwargs={"k": K})

for q in ["docker compose에서 서비스 이름으로 접속하는 이유는?",
          "async 함수 안에서 time.sleep을 쓰면 왜 느려져?",
          "쿠버네티스 파드란 무엇인가?"]:               # 위키에 없는 질문
    print(f"\nQ. {q}")
    for doc, score in vectorstore.similarity_search_with_score(q, k=K):
        preview = doc.page_content[:40].replace("\n", " ")
        print(f"  {score:.4f}  {doc.metadata['id']:<40} {preview}")


Q. docker compose에서 서비스 이름으로 접속하는 이유는?
  0.2740  docker-compose_mingi-3                   - **Q1**: 정확해요. 덧붙이면 compose 파일은 Git에 올라
  0.3101  docker-compose_mingi-0                   # docker compose: 컨테이너 여러 개를 한 번에 (2주차) 
  0.3904  2-fastapi-redis-async_sejin-15           ### 내가 막힌 점  **헷갈렸던 점** - compose 안에서 Re
  0.4358  container-commands_mingi-3               - **Q3 (용어 맞추기)**: 1주차 피드백에서 정리한 것처럼 "설계
  0.4434  dockerfile-instructions_mingi-5          - **CMD**: 맞아요. JSON 배열 형태(exec form)로 쓰

Q. async 함수 안에서 time.sleep을 쓰면 왜 느려져?
  0.3081  async-fastapi_doyu-2                     @app.get("/bad-sleep") async def bad_sle
  0.3251  async-fastapi_mingi-1                    i/o는 결과를 기다리는 작업을 의미한다. Input/Out의 약자  #
  0.3280  when-to-use-async_huiji-0                # async 함수는 언제 필요한가  ## 한 줄 정의 async def
  0.3456  async-fastapi_mingi-2                    - **비유**: 이벤트 루프는 **요리사 1명**이에요. `async 
  0.3466  async-fastapi_mingi-0                    # FastAPI에서 async가 필요한 때 (2주차)  ## 내 측정 



**📝 관찰 메모 (5. 검색)**
- 위키에 없는 질문(쿠버네티스)의 점수는 다른 질문과 비교해 어땠나?
- 같은 페이지의 청크가 여러 개 나왔나? 그게 생성에 좋을까 나쁠까?
-

## 6. 생성 (LCEL 체인 + 출처)

검색한 청크를 프롬프트에 넣어 Gemini가 답하게 하고, 답과 함께 **출처(페이지 이름)**를 돌려줘요.
- `RunnableParallel`로 `context`(검색 결과)와 `question`을 같이 넘기고, `answer`를 덧붙여요.
- 확인: 위키에 있는 질문 1개, 없는 질문 1개 (모른다고 답하는지)

In [29]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY가 없어요. 맨 위 셀을 다시 실행했는지, week3/.env에 키가 있는지 확인하세요"

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash",
                             google_api_key=api_key,
                             temperature=0)

prompt = ChatPromptTemplate.from_template("""너는 팀 스터디 위키를 바탕으로 답하는 도우미야.
아래 문서에 있는 내용만 사용해서 한국어로 답해. 문서에 없으면 "위키에 없는 내용이에요"라고 답해.

[문서]
{context}

[질문]
{question}""")

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['page']}]\n{d.page_content}" for d in docs)

def sources(docs):
    seen = []
    for d in docs:
        if d.metadata["page"] not in seen:
            seen.append(d.metadata["page"])
    return seen

answer_chain = (
    {"context": lambda x: format_docs(x["context"]), "question": lambda x: x["question"]}
    | prompt | llm | StrOutputParser()
)

rag_chain = (
    RunnableParallel(context=retriever, question=RunnablePassthrough())
    .assign(answer=answer_chain)
    .assign(sources=lambda x: sources(x["context"]))
)

In [30]:
def ask(question):
    out = rag_chain.invoke(question)
    print(f"Q. {question}\n")
    print(out["answer"])
    print("\n출처:", ", ".join(out["sources"]))
    return out

_ = ask("docker compose에서 api 컨테이너가 Redis에 접속할 때 주소를 어떻게 써야 해?")

Q. docker compose에서 api 컨테이너가 Redis에 접속할 때 주소를 어떻게 써야 해?

docker compose 환경에서는 `localhost`가 아닌 서비스 이름을 도메인 이름처럼 사용하므로, **`redis://redis:6379`**(또는 `redis://redis:6379/0`)로 써야 합니다. 

컨테이너 입장에서 `localhost`는 자기 자신을 가리키기 때문에 다른 컨테이너의 서비스에 접속하려면 서비스 이름(`redis`)을 사용해야 합니다.

출처: docker-compose_mingi, 2-fastapi-redis-async_sejin


In [31]:
_ = ask("쿠버네티스 파드란 무엇인가?")   # 위키에 없는 질문

Q. 쿠버네티스 파드란 무엇인가?

위키에 없는 내용이에요

출처: volume-port_mingi, vm-conda-docker_mingi, 2-fastapi-redis-async_sejin, docker-compose_mingi


**📝 관찰 메모 (6. 생성)**
- 출처에 나온 페이지가 실제로 답의 근거였나?
- 위키에 없는 질문에 "모른다"고 답했나? 프롬프트의 어떤 줄이 그렇게 만들었나?
-

## 7. 평가 (Recall@5, MRR@5) + 청크 크기 비교

팀 공통 `team/eval/evaluate.py`와 질문셋 25개로 내 retriever를 채점해요.
이어서 청크 크기 200/500/1000을 같은 모델로 비교해요 (함께 실습 리허설). 모델을 다시 불러오지 않도록 `embeddings`를 넘겨요.

In [32]:
sys.path.append(str(REPO / "team" / "eval"))
from evaluate import evaluate, load_questions, build_retriever

questions = load_questions(REPO / "team" / "eval" / "questions.jsonl")
result = evaluate(retriever, questions, k=K, show_misses=True)


질문 25개 | Recall@5 = 1.000 | MRR@5 = 0.823 | 평균 검색 시간 108.7 ms

정답을 1위로 못 찾은 질문
- q01 정답 순위 4 | 비밀 키를 소스 코드에 직접 적어 두면 어떤 위험이 있어?
    정답: ['api-key-env_mingi']  검색: ['1-docker-concepts_sejin', 'docker-compose_mingi', 'redis-ttl-cache_mingi', 'api-key-env_mingi', 'api-key-env_mingi']
- q07 정답 순위 2 | 여러 컨테이너의 실행 설정을 파일 하나에 적어 한 번에 띄우는 방법은?
    정답: ['docker-compose_mingi']  검색: ['1-docker-concepts_sejin', '1-docker-concepts_sejin', 'docker-compose_mingi', '1-docker-concepts_sejin', 'container-commands_mingi']
- q08 정답 순위 2 | api 컨테이너에서 레디스에 접속할 때 주소를 localhost로 쓰면 왜 안 돼?
    정답: ['docker-compose_mingi']  검색: ['2-fastapi-redis-async_sejin', 'docker-compose_mingi', '2-fastapi-redis-async_sejin', '2-fastapi-redis-async_sejin', '2-fastapi-redis-async_sejin']
- q09 정답 순위 3 | 이미지를 만들 때 실행되는 명령과 컨테이너가 시작될 때 실행되는 명령은 어떻게 달라?
    정답: ['dockerfile-instructions_mingi', '1-docker-concepts_sejin']  검색: ['image-vs-container_huiji', 'container-commands_mingi', '1-docker-concepts_sejin', '1-docker-concept

In [33]:
rows = []
for size, overlap in [(200, 20), (500, 50), (1000, 100)]:
    r = build_retriever(WIKI, chunk_size=size, chunk_overlap=overlap, k=K, embeddings=embeddings)
    res = evaluate(r, questions, k=K, verbose=False)
    rows.append({"chunk_size": size, "overlap": overlap,
                 f"Recall@{K}": round(res["recall"], 3), f"MRR@{K}": round(res["mrr"], 3),
                 "평균 검색 ms": round(res["avg_ms"], 1)})
pd.DataFrame(rows)

페이지 15개 -> 청크 295개 (chunk_size=200, overlap=20)
페이지 15개 -> 청크 109개 (chunk_size=500, overlap=50)
페이지 15개 -> 청크 53개 (chunk_size=1000, overlap=100)


,chunk_size,overlap,Recall@5,MRR@5,평균 검색 ms
0,200,20,1.00,0.800,98.2
1,500,50,1.00,0.823,97.7
2,1000,100,0.96,0.873,96.4


**📝 관찰 메모 (7. 평가)**
- 1위로 못 찾은 질문들의 공통점은? (질문이 어려웠나, 청크가 잘못 잘렸나, 페이지가 비슷했나)
- 청크 크기에 따라 점수가 어떻게 바뀌었나? 우리 위키(페이지 1~3KB, 15개)에서 나온 결과를 큰 문서에도 그대로 믿어도 될까?
-